In [ ]:
from tqdm.auto import tqdm
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import pyLDAvis.gensim_models
from gensim import models, corpora
import pyLDAvis
import wikipediaapi
wiki = wikipediaapi.Wikipedia(
    user_agent='MyNLPProject/1.0 (your_email@example.com)',
    language='en'
)
import spacy
spacy_nlp = spacy.load("en_core_web_sm")

__Text mining and processing__: This notebook execute two parts (i) text mining  and (ii) text processing:

- __Text mining__: Documents are created out of text extracted from Wikipedia pages using the wikipedia api and [`Wikipedia-API`](https://pypi.org/project/Wikipedia-API/). This is done by using home built functions defined in the python script [Fetch_Wiki_data.py](Fetch_Wiki_data.py).

- __Text processing__: The text of the documents are then processed using [spaCy](https://spacy.io/) library. That includes
    - tokenization of words,
    - cleaning-up, 
    - lemmatization,
    - POS tagging
    - In the end a dictionary is created, mapping words to numerical ids, and the documents are converted to a bag-of-words format.

In [27]:
!pip install ipywidgets notebook jupyterlab

In [28]:
# Extract members and submembers of the following two categories that are not themselves categories.
cat = wiki.page("Category:Science_fiction_novels_by_year")
cat_years = [page for page in cat.categorymembers.values()]

cat = wiki.page("Category:Science_fiction_novels_by_writer")
cat_writers = [page for page in cat.categorymembers.values()]

pages_by_year = [
    get_categorymembers(c, n_pages_per_level_threshold=1, level=0, max_level=None, verbose=False)
    for c in cat_years
]

pages_by_writer = [
    get_categorymembers(c, n_pages_per_level_threshold=1, level=0, max_level=None, verbose=False)
    for c in cat_writers
]

pages_years = [p for page_list in pages_by_year for p in page_list]
pages_writers = [p for page_list in pages_by_writer for p in page_list]

pages_all = pages_writers + [p for p in pages_years if p not in pages_writers]


years = [int(re.search('\d+', c.title).group()) for c in cat_years]

print(f"There are {len(pages_all)} wikipedia pages (novels) written between {min(years)} and {max(years)}.")

<>:24: DeprecationWarning: invalid escape sequence '\d'
<>:24: DeprecationWarning: invalid escape sequence '\d'
/var/folders/lj/zlfwy44d38zbbqkcdmg7xjk00000gn/T/ipykernel_91091/2000230065.py:24: DeprecationWarning: invalid escape sequence '\d'
  years = [int(re.search('\d+', c.title).group()) for c in cat_years]


There are 5206 wikipedia pages (novels) written between 1805 and 2025.


In [30]:
 # even quick no tqdm 
for p in pages_all:
    documents.append(
        get_page_text(
            p,
            keywords=kws,
            verbose=False,
            use_summary_if_empty=True
        )
    )

__Text processing__: Text is processed with Latent Dirichlet Allocation (LDA) is executed as follows

- The text of each document is tokenized by words.

- Some Stopwords are removed and additional words  are included that occur frequently without being relevant `e.g. 'story', 'character', 'novel'`.

- Using the POS tagging feature of spaCy, only words with the allowed tag are kept. Only `nouns, verbs, adjectives, and adverbs` are use.

- Words are reduce to their lemmas `e.g. the lemma of 'went', 'gone', nd 'goes' is 'go'`

The pages end up containing no tokens if they have no relevant sections, no sections at all those pages are removed from the corpus.

In [31]:
def prepare_text_for_lda(
    document,
    lemmatize=True,
    allowed_postags=None,
    min_len=3,
    additional_stopwords=None
):
    """
    Returns tokens or lemmas for a document,
    filtering stopwords.
    """

    if additional_stopwords is None:
        additional_stopwords = []

    if document == []:
        return []

    tokens = []

    for section in document:
        tokens += [
            word for word in spacy_nlp(section)
            if (
                len(word.text) >= min_len
                and not word.is_stop
                and word.text not in additional_stopwords
            )
        ]

    if allowed_postags is not None:
        tokens = [
            t for t in tokens
            if t.pos_ in allowed_postags
        ]

    if lemmatize:
        tokens = [
            t.lemma_.lower()
            for t in tokens
        ]
    else:
        tokens = [
            t.text.lower()
            for t in tokens
        ]

    return [
        t for t in tokens
        if t not in additional_stopwords
    ]

In [25]:
additional_stopwords = [
    'story','character','novel','book','write',
    'writer','fiction','series','publish','year',
    'television','feature','american','british',
    'narrator','original','reference','author',
    'chapter','film','episode','release'
]

indices_empty_documents = []
tokenized_data = []

for i, doc in tqdm(enumerate(documents), total=len(documents)):
    
    lda_tokens = prepare_text_for_lda(
        doc,
        lemmatize=True,
        allowed_postags=['NOUN','VERB'],
        min_len=3,
        additional_stopwords=additional_stopwords
    )

    if len(lda_tokens) == 0:
        indices_empty_documents.append(i)

    tokenized_data.append(lda_tokens)

print(f"{len(indices_empty_documents)} pages were ignored.")

pages = [
    p for i,p in enumerate(pages_all)
    if i not in indices_empty_documents
]

tokenized_data = [
    t for i,t in enumerate(tokenized_data)
    if i not in indices_empty_documents
]

print(f"There are {len(tokenized_data)} documents.")

100%|██████████| 5410/5410 [07:24<00:00, 12.16it/s]


5 pages were ignored.
There are 5405 documents.


In [32]:
# Build a Dictionary - associate a numeric id to each word
dictionary = corpora.Dictionary(tokenized_data)
 
# Transform the collection of texts to a numerical form (bag-of-words)
corpus = [dictionary.doc2bow(text) for text in tokenized_data]

__Additional text mining__: Here, the wikipedia pages are  mine in order to obtain information about the __author__ of each page. The wikipedia API is use  to extract the information in the __infoboxes__ of the wikipedia pages box on the topright part of a page. Currently, if a page does not have the name of author in its info box, or if it does not contain an info box, the author is left unspecified. This could be improved by mining the information in the main text of the page.

In [34]:
import requests
import time

def get_authors(set_of_pages):

    pageids = [p.pageid for p in set_of_pages]

    PARAMS = {
        "action":"query",
        "prop":"revisions",
        "rvprop":"user",
        "format":"json",
        "pageids":"|".join(map(str,pageids))
    }

    URL = "https://en.wikipedia.org/w/api.php"

    for attempt in range(3):
        try:
            r = requests.get(
                URL,
                params=PARAMS,
                timeout=30,
                headers={
                    "User-Agent":
                    "MyNLPProject/1.0 your_email@example.com"
                }
            )

            r.raise_for_status()

            data = r.json()
            break

        except Exception as e:
            print(f"Retry {attempt+1}: {e}")
            time.sleep(2)

    else:
        return ['NA'] * len(set_of_pages)

    authors = []

    for pid in pageids:
        try:
            author = data["query"]["pages"][str(pid)]["revisions"][0]["user"]
        except:
            author = "NA"

        authors.append(author)

    return authors

In [37]:
%whos

Variable                  Type             Data/Info
----------------------------------------------------
N                         int              50
additional_stopwords      list             n=22
authors                   list             n=0
cat                       WikipediaPage    <object with id 5137619792 (str() failed)>
cat_writers               list             n=94
cat_years                 list             n=158
construct_author2doc      function         <function construct_author2doc at 0x1799e74c0>
corpora                   module           <module 'gensim.corpora' <...>sim/corpora/__init__.py'>
corpus                    list             n=5405
dictionary                Dictionary       Dictionary<18916 unique t<...>dge', 'acquaintance']...>
doc                       list             n=1
documents                 list             n=6285
f                         TextIOWrapper    <_io.TextIOWrapper name='<...>ode='w' encoding='UTF-8'>
get_authors               function  

In [ ]:
#author2doc = construct_author2doc(doc2author)

In [ ]:
#print(type(author2doc))
#print(len(author2doc))

In [38]:
from gensim.models.atmodel import construct_author2doc
doc2author = {
    i: [f"author_{i}"]
    for i in range(len(tokenized_data))
}
author2doc = construct_author2doc(doc2author)
print("author2doc created")

author2doc created


In [39]:
import os, json
os.makedirs("data", exist_ok=True)

with open("data/author2doc.json", "w") as f:
    json.dump(
        {str(k):v for k,v in author2doc.items()},
        f,
        indent=2
    )

with open("data/tokenized_data.json", "w") as f:
    json.dump(tokenized_data, f, indent=2)

dictionary.save("data/dictionary")
corpora.MmCorpus.serialize(
    "data/corpus.mm",
    corpus
)